# Notebook 01b — MiniFASNetV2 pretrained Vanilla vs CSMR, 20K scale confirmation

Uses the frozen 00b-v2 pretrained recipe after its values are transcribed into the config cell. The official PAD-pretrained feature weights and deterministic fresh classifier initialize both branches identically. Test-5K is fixed descriptive evaluation; LCC is external-development stress; CASIA remains untouched. No ONNX export or tuning occurs here.


In [ ]:
# CONFIG / DATA ROOTS: freeze before running. Mount Notebook 00 output as FINAL_RESOURCE_DIR.
CELEBA_ROOT=("/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/"
             "CelebA_Spoof_/CelebA_Spoof")
LCC_FASD_ROOT="/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD"
FINAL_RESOURCE_DIR="/kaggle/input/datasets/maithanhphuong/final-resources-v1/minifasnet_final_resources_v1"
MINIFASNET_SOURCE_PATH="/kaggle/input/datasets/maithanhphuong/minifasnet/MiniFASNet.py"
MINIFASNET_CHECKPOINT_PATH="/kaggle/input/datasets/maithanhphuong/minifasnet/2.7_80x80_MiniFASNetV2.pth"
OUTPUT_ROOT='/kaggle/working/minifasnet_scale20k_pretrained_seed100'
TRAIN_SEED=100; DATA_SEED=42
TUNING_CONFIG_SOURCE='00b-minifasnetv2-5k-hparam-tuning_v2/selection/frozen_hparams_v2.json (values transcribed after freeze)'
TUNING_CONFIG_SHA256='bfc8652b237d7a8156bf455667bcbc00794baa9b0f0283c4a449406da6caeb0a'
SCRATCH_BACKBONE_LR=0.001; SCRATCH_CLASSIFIER_LR=0.001
PRETRAINED_BACKBONE_LR=0.0002; PRETRAINED_CLASSIFIER_LR=0.001
METRIC_TIE_EPS=1e-12; METRIC_VERIFY_EPS=1e-6
BATCH_SIZE=128 # If CUDA OOM, restart BOTH branches with 64. No branch-only change.
NUM_WORKERS=4; AMP=True; RESUME=False
MAX_EPOCHS=8; EARLY_STOP_PATIENCE=2; EARLIEST_EARLY_STOP_EPOCH=4
WEIGHT_DECAY=1e-4; LABEL_SMOOTHING=.1; GRAD_CLIP=5.
BACKBONE_LR=PRETRAINED_BACKBONE_LR; CLASSIFIER_LR=PRETRAINED_CLASSIFIER_LR
CROP_SCALE=2.7; INPUT_SIZE=80
CSMR_WARMUP_EPOCHS=1; CSMR_LAMBDA=0.4; CSMR_BAND_CENTERS=[.15,.45,.75]
CSMR_BAND_SIGMA=.1; CSMR_GAIN_RANGE=[0.65, 0.9]; CSMR_PRESERVE_DC=True; SELECTION_CHUNK_SIZE=32
TRAIN_MANIFEST='scale_train20k'; VAL_MANIFEST='scale_val5k'
TRAIN_EXPECTED=20000; VAL_EXPECTED=5000
INIT_NAME='pretrained'
TEST_ROLE='fixed in-domain descriptive evaluation during pretrained 20K confirmation'


In [ ]:
# Complete these frozen values from the executed 00b-v2 output before any training.
if not isinstance(TUNING_CONFIG_SHA256,str) or len(TUNING_CONFIG_SHA256)!=64 or any(c not in '0123456789abcdef' for c in TUNING_CONFIG_SHA256.lower()):
    raise RuntimeError('Set TUNING_CONFIG_SHA256 from selection/frozen_hparams_v2.sha256 after 00b-v2 runs')
if TUNING_CONFIG_SHA256=='2aedb0477c6d0ed05c9a96ba8918497b312fdebd0f820c1170e4c2b56398155e':
    raise RuntimeError('This is the old v1 freeze SHA, not the regime-specific v2 freeze')
if INIT_NAME=='pretrained':
    if CSMR_LAMBDA is None or CSMR_GAIN_RANGE is None:
        raise RuntimeError('Set pretrained CSMR_LAMBDA and CSMR_GAIN_RANGE from frozen_hparams_v2.json after 00b-v2 runs')
    if CSMR_LAMBDA not in (.10,.20,.40) or CSMR_GAIN_RANGE not in ([.75,.95],[.65,.90],[.55,.85]):
        raise RuntimeError('Pretrained CSMR values are outside frozen tuning grid')


In [ ]:
import csv, hashlib, importlib.util, json, math, os, random, shutil, time, zipfile, copy, gc
from collections import Counter, OrderedDict
from pathlib import Path, PurePosixPath
import cv2, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import confusion_matrix, roc_curve
from IPython.display import FileLink, display
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


def assert_ordered_keys(actual, expected, name):
    actual = list(map(str, actual))
    expected = list(map(str, expected))
    if actual != expected:
        raise RuntimeError(
            "{} sample keys/order differ from the frozen manifest: actual_n={}, expected_n={}".format(
                name, len(actual), len(expected)
            )
        )


def bbox_jitter(record):
    x1, y1, x2, y2 = map(float, record["bbox_xyxy"])
    w, h = x2 - x1, y2 - y1
    if random.random() < 0.20:
        base = max(w, h)
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        scale = random.uniform(0.95, 1.05)
        cx += random.uniform(-0.05, 0.05) * base
        cy += random.uniform(-0.05, 0.05) * base
        w, h = w * scale, h * scale
        return [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]
    return [x1, y1, x2, y2]


def apply_e1_mild_augmentation(rgb):
    if random.random() < 0.5:
        rgb = cv2.flip(rgb, 1)
    if random.random() < 0.30:
        alpha = random.uniform(0.90, 1.10)
        beta = random.uniform(-0.10, 0.10) * 255.0
        rgb = np.clip(rgb.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
    return rgb


def binary_score(logits):
    logits = torch.as_tensor(logits)
    if logits.ndim != 2 or logits.shape[1] != 3:
        raise ValueError("PAD logits must have shape [N, 3] for Real/Physical/Digital")
    if not torch.isfinite(logits).all():
        raise ValueError("PAD logits contain NaN or Infinity")
    return logits[:, 0] - torch.logsumexp(logits[:, 1:], dim=1)


def safe_auc(y_real, scores):
    from sklearn.metrics import roc_auc_score
    y_real = np.asarray(y_real, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y_real.ndim != 1 or scores.ndim != 1 or len(y_real) != len(scores) or len(y_real) == 0:
        raise ValueError("AUC needs non-empty one-dimensional labels and scores of equal length")
    if not np.isfinite(scores).all():
        raise ValueError("AUC scores contain NaN or Infinity")
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("AUC requires both binary classes (positive class is Real)")
    return float(roc_auc_score(y_real, scores))


def metrics_binary(y3, scores, threshold, coverage=None, candidate_n=None):
    y3 = np.asarray(y3, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    if y3.ndim != 1 or scores.ndim != 1 or len(y3) != len(scores) or len(y3) == 0:
        raise ValueError("PAD metrics need non-empty 1-D labels/scores with equal lengths")
    if not set(map(int, np.unique(y3))).issubset({0, 1, 2}):
        raise ValueError("PAD labels must be from the frozen class mapping 0/1/2")
    if not np.isfinite(scores).all() or not np.isfinite(float(threshold)):
        raise ValueError("PAD scores and threshold must be finite")
    y_real = (y3 == 0).astype(np.int64)
    if set(map(int, np.unique(y_real))) != {0, 1}:
        raise ValueError("PAD metrics require at least one Real and one Attack sample")
    if coverage is not None and (not np.isfinite(float(coverage)) or not 0.0 <= float(coverage) <= 1.0):
        raise ValueError("Detector coverage must be finite and within [0, 1]")
    scored_n = len(y3)
    candidate_n = int(candidate_n if candidate_n is not None else scored_n)
    if candidate_n < scored_n:
        raise ValueError("candidate_n cannot be less than n_scored")
    pred_real = scores >= float(threshold)
    real = y_real == 1
    attack = y_real == 0
    bpcer = float(np.mean(~pred_real[real]))
    apcer = float(np.mean(pred_real[attack]))
    acer = 0.5 * (apcer + bpcer)
    return {
        "n_scored": int(scored_n),
        "candidate_n": candidate_n,
        "detector_coverage": float(coverage if coverage is not None else 1.0),
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "hter": acer,
        "auc": safe_auc(y_real, scores),
        "accuracy": float(np.mean(pred_real == y_real)),
        "locked_logit_threshold": float(threshold),
    }


def metrics_celeba(y3, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    result = metrics_binary(y3, scores, threshold, coverage=1.0, candidate_n=len(y3))
    result["accuracy_3class"] = float(np.mean(np.argmax(logits, axis=1) == np.asarray(y3)))
    return result, scores


def make_prediction_frame(keys, y3, codes, logits, threshold):
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    pred_real = scores >= threshold
    result = pd.DataFrame({
        "key": list(map(str, keys)),
        "true_class": np.asarray(y3, dtype=np.int64),
        "attack_code": np.asarray(codes, dtype=np.int64),
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
        "pred_real": pred_real,
        "correct_binary": pred_real == (np.asarray(y3) == 0),
        "pred_class_3": np.argmax(logits, axis=1),
        "correct_3class": np.argmax(logits, axis=1) == np.asarray(y3),
    })
    return result


def eval_spatial(model, loader, device, amp):
    model.eval()
    logits_all, y_all, keys_all, codes_all = [], [], [], []
    with torch.no_grad():
        for x, y, keys, codes in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
            codes_all.extend(np.asarray(codes, dtype=np.int64).tolist())
    if not logits_all:
        raise RuntimeError("Spatial evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    labels = np.asarray(y_all, dtype=np.int64)
    codes = np.asarray(codes_all, dtype=np.int64)
    if len(logits) != len(labels) or len(keys_all) != len(labels) or len(codes) != len(labels):
        raise RuntimeError("Spatial evaluation outputs have inconsistent sample counts")
    return logits, labels, list(map(str, keys_all)), codes


def eval_lcc_spatial(model, loader, device, amp, lcc_frame, threshold):
    model.eval()
    logits_all, y_all, keys_all = [], [], []
    with torch.no_grad():
        for x, y, keys, _status in loader:
            x = x.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=amp):
                logits = model(x)
            logits_all.append(logits.float().cpu().numpy())
            y_all.extend(np.asarray(y, dtype=np.int64).tolist())
            keys_all.extend(list(keys))
    if not logits_all:
        raise RuntimeError("Common LCC VALID evaluation loader produced no samples")
    logits = np.concatenate(logits_all)
    y = np.asarray(y_all, dtype=np.int64)
    keys = list(map(str, keys_all))
    expected_paths = lcc_frame.loc[lcc_frame["status"].astype(str) == "VALID", "path"].astype(str).tolist()
    assert_ordered_keys(keys, expected_paths, "Stage 1 LCC")
    if len(logits) != len(y) or len(keys) != len(y):
        raise RuntimeError("Stage 1 LCC logits/labels/keys have inconsistent counts")
    scores = binary_score(torch.as_tensor(logits)).cpu().numpy()
    scored = pd.DataFrame({
        "path": keys, "label": y,
        "logit_real": logits[:, 0],
        "logit_physical": logits[:, 1],
        "logit_digital": logits[:, 2],
        "pad_logit_score": scores,
    })
    full = lcc_frame[["path", "label", "status"]].merge(
        scored, on=["path", "label"], how="left", validate="one_to_one"
    )
    full["dataset_role"] = "LCC-FASD external-dev / cross-domain stress set"
    valid = full["status"].astype(str).eq("VALID")
    if full.loc[valid, "pad_logit_score"].isna().any() or int(valid.sum()) != len(keys):
        raise RuntimeError("Stage 1 inference failed to score the exact common LCC VALID set")
    full["pred_real"] = np.nan
    full.loc[valid, "pred_real"] = full.loc[valid, "pad_logit_score"] >= float(threshold)
    full["pred_real"] = full["pred_real"].where(valid, np.nan)
    return logits, y, keys, full


def write_zip(folder, zip_path):
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and path.resolve() != Path(zip_path).resolve():
                archive.write(path, arcname=Path(Path(folder).name) / path.relative_to(folder))


def strict_json_write(path,payload):
    Path(path).write_text(json.dumps(payload,indent=2,allow_nan=False)+'\n')

def strict_json_load(path):
    return json.loads(Path(path).read_text(),parse_constant=lambda s: (_ for _ in ()).throw(ValueError(s)))

def ordered_sha(keys):return sha256_text('\n'.join(map(str,keys)))

def threshold_from_val(y3,scores):
    y=np.asarray(y3,dtype=np.int64);s=np.asarray(scores,dtype=np.float64)
    if len(y)!=len(s) or not len(y) or not np.isfinite(s).all() or set(np.unique(y))!={0,1,2}:
        raise ValueError('Val threshold needs finite 3-class labels/scores')
    order=np.argsort(s,kind='mergesort');v=s[order];real=(y[order]==0)
    nr=int(real.sum());na=len(y)-nr;bound=np.r_[0,np.flatnonzero(np.diff(v))+1,len(v)]
    rp=np.r_[0,np.cumsum(real)];ap=np.r_[0,np.cumsum(~real)];pairs=[]
    for j,k in enumerate(bound):
        t=float(np.nextafter(v[0],-np.inf)) if j==0 else (float(np.nextafter(v[-1],np.inf)) if j==len(bound)-1 else float(v[k-1]/2+v[k]/2))
        pairs.append((.5*(rp[k]/nr+(na-ap[k])/na),abs(t),t))
    return min(pairs)[2]

def resource_tree_sha(root):
    h=hashlib.sha256()
    for p in sorted(Path(root).rglob('*')):
        if p.is_file():h.update(str(p.relative_to(root)).encode()+b'\0'+sha256_file(p).encode()+b'\n')
    return h.hexdigest()
def mini_crop_bgr(image,bbox_xyxy,scale=CROP_SCALE,size=INPUT_SIZE):
    # Official CropImage._get_new_box geometry, adapted from xyxy face coordinates.
    src_h,src_w=image.shape[:2]
    x1,y1,x2,y2=map(float,bbox_xyxy)
    box_w,box_h=x2-x1,y2-y1
    if box_w<=0 or box_h<=0 or src_w<2 or src_h<2:
        raise ValueError('Invalid MiniFASNet image/bbox geometry')
    used_scale=min((src_h-1)/box_h,(src_w-1)/box_w,float(scale))
    new_w,new_h=box_w*used_scale,box_h*used_scale
    center_x,center_y=box_w/2+x1,box_h/2+y1
    left,top=center_x-new_w/2,center_y-new_h/2
    right,bottom=center_x+new_w/2,center_y+new_h/2
    if left<0: right-=left;left=0
    if top<0: bottom-=top;top=0
    if right>src_w-1: left-=right-src_w+1;right=src_w-1
    if bottom>src_h-1: top-=bottom-src_h+1;bottom=src_h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=src_w or by>=src_h or rx<lx or by<ty:
        raise RuntimeError(f'MiniFASNet crop escaped image: {(lx,ty,rx,by)}')
    patch=image[ty:by+1,lx:rx+1]
    if patch.size==0:raise RuntimeError('Empty MiniFASNet crop')
    return cv2.resize(patch,(size,size)),(lx,ty,rx,by),used_scale


def mini_tensor(patch):
    if patch.shape!=(INPUT_SIZE,INPUT_SIZE,3) or patch.dtype!=np.uint8:
        raise ValueError('Expected 80x80 uint8 BGR patch')
    return torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).float().div_(255.)


def source_module(path):
    spec=importlib.util.spec_from_file_location('local_minifasnet_v2',path)
    if spec is None or spec.loader is None:raise RuntimeError('Cannot import local MiniFASNet.py')
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
    return module


def new_model(module):
    return module.MiniFASNetV2(embedding_size=128,conv6_kernel=(5,5),
        drop_p=.2,num_classes=3,img_channel=3)


def state_fingerprint(state):
    h=hashlib.sha256()
    for key in sorted(state):
        value=state[key].detach().cpu().contiguous()
        h.update(key.encode());h.update(str(value.dtype).encode());h.update(str(tuple(value.shape)).encode())
        h.update(value.numpy().tobytes())
    return h.hexdigest()


def load_pretrained_canonical(module,checkpoint_path):
    raw=torch.load(checkpoint_path,map_location='cpu',weights_only=True)
    if not isinstance(raw,OrderedDict) or not raw or not all(k.startswith('module.') for k in raw):
        raise RuntimeError('Expected official OrderedDict with module. prefix')
    stripped=OrderedDict((k[7:],v) for k,v in raw.items())
    if list(stripped['prob.weight'].shape)!=[3,128]:raise RuntimeError('Official prob.weight shape differs from [3,128]')
    if tuple(stripped['conv_6_dw.conv.weight'].shape[-2:])!=(5,5):raise RuntimeError('Official conv6 kernel is not 5x5')
    seed_everything(TRAIN_SEED)
    model=new_model(module)
    model_keys=model.state_dict()
    skipped=['prob.weight']
    shape_mismatch=[k for k,v in stripped.items() if k in model_keys and tuple(v.shape)!=tuple(model_keys[k].shape) and k not in skipped]
    unexpected=[k for k in stripped if k not in model_keys]
    missing_pretrained=[k for k in model_keys if k not in stripped and k not in skipped]
    if shape_mismatch or unexpected or missing_pretrained:raise RuntimeError(f'Unrelated checkpoint mismatch: shape={shape_mismatch[:5]} unexpected={unexpected[:5]} missing={missing_pretrained[:5]}')
    compatible=OrderedDict((k,v) for k,v in stripped.items() if k not in skipped)
    missing,unexpected_load=model.load_state_dict(compatible,strict=False)
    if set(missing)!={'prob.weight'} or unexpected_load:raise RuntimeError(f'Loading audit failed: missing={missing}, unexpected={unexpected_load}')
    model.prob=nn.Linear(128,3,bias=False)  # new task semantics, deterministic seed 100
    if not model.prob.weight.requires_grad or not model.conv1.conv.weight.requires_grad:
        raise RuntimeError('Classifier/backbone must remain trainable')
    audit={'schema':'mini_fasnet_pretrained_loading_audit_v1','checkpoint_file':checkpoint_path.name,
        'checkpoint_sha256':sha256_file(checkpoint_path),'checkpoint_bytes':checkpoint_path.stat().st_size,
        'architecture_source_sha256':sha256_file(MINIFASNET_SOURCE),
        'checkpoint_keys':len(raw),'loaded_pretrained_key_count':len(compatible),
        'missing_keys':list(missing),'unexpected_keys':list(unexpected_load),
        'intentionally_skipped_keys':skipped,'shape_mismatch_keys':shape_mismatch,
        'input_size':[80,80],'crop_scale':CROP_SCALE,'channel_order':'BGR',
        'conv6_kernel':[5,5],'embedding_size':128,'classifier_shape':[3,128],
        'total_params':sum(p.numel() for p in model.parameters()),
        'trainable_params':sum(p.numel() for p in model.parameters() if p.requires_grad)}
    return model,audit
import torch.nn.functional as F
BANDS=("LOW","MID","HIGH")
def normalize_unit(x):
    # MiniFASNet checkpoint expects BGR float [0,1]; no MNV3 mean/std.
    return x


def radial_masks(size=INPUT_SIZE, device='cpu'):
    y = torch.arange(size, device=device, dtype=torch.float32) - size // 2
    yy, xx = torch.meshgrid(y, y, indexing='ij')
    radius = torch.sqrt(xx.square() + yy.square()) / math.sqrt(2 * (size // 2) ** 2)
    centers = torch.as_tensor(CSMR_BAND_CENTERS, dtype=torch.float32, device=device)
    masks = torch.exp(-((radius[None] - centers[:, None, None]) ** 2) /
                      (2 * CSMR_BAND_SIGMA ** 2))
    if masks.shape != (3, size, size) or not torch.isfinite(masks).all() or not ((masks >= 0) & (masks <= 1)).all():
        raise RuntimeError('Invalid CSMR radial masks')
    if len(set(map(float, CSMR_BAND_CENTERS))) != 3:
        raise RuntimeError('CSMR band centers must differ')
    return masks, radius


def spectral_views(x, gains, masks):
    if x.ndim != 4 or x.shape[1:] != (3, INPUT_SIZE, INPUT_SIZE) or not torch.isfinite(x).all():
        raise ValueError('CSMR input must be finite [B,3,80,80]')
    x = x.float()
    gains = gains.to(device=x.device, dtype=torch.float32).reshape(-1)
    if len(gains) != len(x) or not ((gains >= 0) & (gains <= 1)).all():
        raise ValueError('Invalid gains')
    spectrum = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2, -1))
    gain_fields = 1 - (1 - gains[:, None, None, None]) * masks[None]
    if CSMR_PRESERVE_DC:
        gain_fields[:, :, INPUT_SIZE // 2, INPUT_SIZE // 2] = 1.0
    perturbed = spectrum[:, None] * gain_fields[:, :, None]
    raw = torch.fft.ifft2(torch.fft.ifftshift(perturbed, dim=(-2, -1)), norm='ortho').real
    clipped = ((raw < 0) | (raw > 1)).float().mean(dim=(2, 3, 4))
    views = raw.clamp(0, 1)
    if not torch.isfinite(views).all():
        raise RuntimeError('Non-finite CSMR intervention')
    return views, gain_fields, clipped


def aligned_margin(logits, labels):
    d = binary_score(logits.float())
    labels = torch.as_tensor(labels, device=d.device)
    if not torch.isin(labels, torch.tensor([0, 1, 2], device=d.device)).all():
        raise ValueError('Invalid PAD label')
    return torch.where(labels == 0, d, -d)


def harmful_margin_loss(clean_margin, worst_margin):
    drop = clean_margin.detach() - worst_margin
    harmful = drop > 0
    per = F.smooth_l1_loss(worst_margin, clean_margin.detach(), reduction='none')
    return (per * harmful.float()).mean(), harmful, drop


def selection_by_margins(margins):
    if margins.ndim != 2 or margins.shape[1] != 3 or not torch.isfinite(margins).all():
        raise ValueError('Expected finite [B,3] margins')
    return margins.argmin(dim=1)


def select_worst_view(model, base, labels, masks, amp, chunk_size):
    if chunk_size < 1:
        raise ValueError('Selection chunk size must be positive')
    was_training = model.training
    model.eval()
    chosen, indexes, all_gains, all_clips = [], [], [], []
    try:
        with torch.no_grad():
            for start in range(0, len(base), chunk_size):
                x = base[start:start+chunk_size]
                y = labels[start:start+chunk_size]
                gains = torch.empty(len(x), device=x.device).uniform_(*CSMR_GAIN_RANGE)
                views, _, clips = spectral_views(x, gains, masks)
                margins = []
                for band in range(3):
                    with torch.cuda.amp.autocast(enabled=amp):
                        logits = model(normalize_unit(views[:, band]))
                    margins.append(aligned_margin(logits, y))
                indexes_chunk = selection_by_margins(torch.stack(margins, dim=1))
                ix = torch.arange(len(x), device=x.device)
                chosen.append(views[ix, indexes_chunk].detach())
                indexes.append(indexes_chunk.detach())
                all_gains.append(gains.detach())
                all_clips.append(clips[ix, indexes_chunk].detach())
    finally:
        model.train(was_training)
    return tuple(torch.cat(a) for a in (chosen, indexes, all_gains, all_clips))


def intervention_sanity(masks, device):
    gen = torch.Generator(device=device).manual_seed(TRAIN_SEED)
    x = torch.rand((2, 3, INPUT_SIZE, INPUT_SIZE), generator=gen, device=device)
    gain = torch.tensor([0.65, 0.90], device=device)
    views, fields, clips = spectral_views(x, gain, masks)
    assert views.shape == (2, 3, 3, INPUT_SIZE, INPUT_SIZE)
    assert torch.all(fields[:, :, INPUT_SIZE//2, INPUT_SIZE//2] == 1)
    unity, _, _ = spectral_views(x, torch.ones(2, device=device), masks)
    assert torch.max(torch.abs(unity[:, 0] - x)).item() < 1e-5
    ft = torch.fft.fftshift(torch.fft.fft2(x, norm='ortho'), dim=(-2,-1))
    modified = ft[:, None] * fields[:, :, None]
    valid = ft.abs()[:, None] > 1e-5
    phase_error = (torch.angle(modified) - torch.angle(ft[:, None])).abs()
    phase_error = torch.minimum(phase_error, 2*math.pi-phase_error)
    assert phase_error.expand_as(valid.expand_as(modified))[valid.expand_as(modified)].max().item() < 1e-4
    report = {}
    for i, band in enumerate(BANDS):
        diff = (views[:, i] - x).abs()
        report[band] = {'mean_absolute_pixel_difference': float(diff.mean()),
                        'max_absolute_pixel_difference': float(diff.max()),
                        'clipped_pixel_fraction': float(clips[:, i].mean())}
        if report[band]['clipped_pixel_fraction'] > 0.05:
            print('WARNING: clipping exceeds 5% for', band)
    return report
class FinalCelebADataset(Dataset):
    def __init__(self,frame,root,records,train=False):
        self.frame=frame.reset_index(drop=True);self.root=Path(root);self.records=records;self.train=train
        missing=set(self.frame.sample_key.astype(str))-set(records)
        if missing:raise RuntimeError(f'CelebA bbox missing {len(missing)} manifest keys')
    def __len__(self):return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i];key=str(row.sample_key);rec=self.records[key]
        if rec.get('status') not in ('VALID','FALLBACK_DET_FAIL','FALLBACK_SUSPICIOUS') or 'crop_factor' in rec:
            raise RuntimeError('Invalid localization-only bbox: '+key)
        image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        bbox=bbox_jitter(rec) if self.train else rec['bbox_xyxy']
        patch,_,_=mini_crop_bgr(image,bbox)
        if self.train:patch=apply_e1_mild_augmentation(patch)
        return mini_tensor(patch),int(row.three_class_label),key,int(row.attack_code)

class FinalLCCDataset(Dataset):
    def __init__(self,frame,root,records):
        self.rows=[];self.root=Path(root)
        for row in frame.itertuples(index=False):
            if str(row.status)=='VALID':
                rec=records[str(row.path)]
                if rec.get('status')!='VALID' or 'crop_factor' in rec:raise RuntimeError('LCC localization record invalid')
                self.rows.append((str(row.path),int(row.label),rec))
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        key,label,rec=self.rows[i];image=cv2.imread(str(self.root/key),cv2.IMREAD_COLOR)
        if image is None:raise FileNotFoundError(self.root/key)
        patch,_,_=mini_crop_bgr(image,rec['bbox_xyxy'])
        return mini_tensor(patch),label,key,'VALID'


In [ ]:
root=Path(FINAL_RESOURCE_DIR).expanduser()
if not (root/'config.json').is_file():raise FileNotFoundError('Set FINAL_RESOURCE_DIR to exact frozen resource directory: '+str(root))
if not (root/'config.json').is_file():raise FileNotFoundError('Notebook00 frozen resource bundle: '+str(root))
output=Path(OUTPUT_ROOT);vanilla_dir=output/'vanilla';csmr_dir=output/'csmr';comparison_dir=output/'comparison'
for folder in (vanilla_dir,csmr_dir,comparison_dir):folder.mkdir(parents=True,exist_ok=True)
celeba_root=Path(CELEBA_ROOT);lcc_root=Path(LCC_FASD_ROOT)
if not celeba_root.is_dir() or not lcc_root.is_dir():raise FileNotFoundError('Set explicit CelebA/LCC roots')
MINIFASNET_SOURCE=Path(MINIFASNET_SOURCE_PATH)
if not MINIFASNET_SOURCE.is_file():raise FileNotFoundError(MINIFASNET_SOURCE)
manifest_fp=strict_json_load(root/'manifests/manifest_fingerprints.json')
resource_provenance=strict_json_load(root/'provenance.json')
resource_config=strict_json_load(root/'config.json')
if resource_config['input']!={'scale':2.7,'size':[80,80],'channels':'BGR','range':'[0,1]'}:
    raise RuntimeError('Frozen preprocessing differs')
frames={}
for name,fp in manifest_fp.items():
    path=root/'manifests'/(name+'.csv')
    if sha256_file(path)!=fp['file_sha256']:raise RuntimeError('Manifest SHA mismatch: '+name)
    frame=pd.read_csv(path)
    if len(frame)!=fp['sample_count'] or ordered_sha(frame.sample_key.astype(str))!=fp['ordered_sample_key_sha256']:
        raise RuntimeError('Manifest fingerprint mismatch: '+name)
    if set(frame.three_class_label.astype(int))!={0,1,2}:raise RuntimeError('Manifest class missing: '+name)
    frames[name]=frame
full_train,full_val,test=frames['final_train100k'],frames['final_val15k'],frames['final_test5k']
assert (len(full_train),len(full_val),len(test))==(100000,15000,5000)
assert set(frames['scale_train20k'].sample_key)<=set(full_train.sample_key)
assert set(frames['scale_val5k'].sample_key)<=set(full_val.sample_key)
assert not set(full_train.sample_key)&set(full_val.sample_key)
assert not (set(full_train.sample_key)|set(full_val.sample_key))&set(test.sample_key)
assert set(full_train.subject_id.astype(str)).isdisjoint(set(full_val.subject_id.astype(str)))
assert set(full_train.subject_id.astype(str)).isdisjoint(set(test.subject_id.astype(str)))
train_frame=frames[TRAIN_MANIFEST];val_frame=frames[VAL_MANIFEST]
assert len(train_frame)==TRAIN_EXPECTED and len(val_frame)==VAL_EXPECTED
bbox_path=root/'bbox/celeba_final_bbox_cache.json';lcc_bbox_path=root/'bbox/lcc_bbox_cache.json'
lcc_path=root/'manifests/lcc_external_dev.csv'
for path,expected in [(bbox_path,resource_provenance['celeba_bbox_sha256']),
    (lcc_bbox_path,resource_provenance['lcc_bbox_sha256']),
    (lcc_path,resource_provenance['lcc_manifest_sha256'])]:
    if sha256_file(path)!=expected:raise RuntimeError('Frozen resource SHA mismatch: '+str(path))
bbox_cache=strict_json_load(bbox_path)['records'];lcc_cache=strict_json_load(lcc_bbox_path)['records']
filter_audit_path=root/'bbox/train_filter_audit.json'
if sha256_file(filter_audit_path)!=resource_provenance['train_filter_audit_sha256']:
    raise RuntimeError('Frozen Train filter audit SHA mismatch')
train_filter_audit=strict_json_load(filter_audit_path)
if train_filter_audit['final_sampled_train_count']!=100000:
    raise RuntimeError('Train100K hygiene audit incomplete')
for key in full_train.sample_key.astype(str):
    rec=bbox_cache[key]
    if rec.get('status')=='FALLBACK_SUSPICIOUS' or rec.get('status') not in ('VALID','FALLBACK_DET_FAIL'):
        raise RuntimeError('Frozen Train contains disallowed bbox status: '+key)
    x1,y1,x2,y2=map(float,rec['bbox_xyxy'])
    if not all(np.isfinite((x1,y1,x2,y2))) or min(x2-x1,y2-y1)<48:
        raise RuntimeError('Frozen Train contains small/invalid face bbox: '+key)

lcc_frame=pd.read_csv(lcc_path,keep_default_na=False)
assert set(bbox_cache)==set(full_train.sample_key)|set(full_val.sample_key)|set(test.sample_key)
assert set(lcc_cache)==set(lcc_frame.path.astype(str))
assert set(lcc_frame[lcc_frame.status=='VALID'].label.astype(int))=={0,1}
if any('crop_factor' in r for r in bbox_cache.values()) or any('crop_factor' in r for r in lcc_cache.values()):
    raise RuntimeError('Bbox cache contains model crop factor')
for frame in (train_frame,val_frame,test):
    if not all((celeba_root/k).is_file() for k in frame.sample_key.astype(str)):
        raise FileNotFoundError('CelebA manifest path missing')
if not all((lcc_root/p).is_file() for p in lcc_frame.path.astype(str)):raise FileNotFoundError('LCC image missing')
resource_hashes={'bundle_tree_sha256':resource_tree_sha(root),'train_manifest_sha256':sha256_file(root/'manifests'/(TRAIN_MANIFEST+'.csv')),
    'val_manifest_sha256':sha256_file(root/'manifests'/(VAL_MANIFEST+'.csv')),
    'test_manifest_sha256':sha256_file(root/'manifests/final_test5k.csv'),
    'lcc_manifest_sha256':sha256_file(lcc_path),'celeba_bbox_sha256':sha256_file(bbox_path),
    'lcc_bbox_sha256':sha256_file(lcc_bbox_path),'minifasnet_source_sha256':sha256_file(MINIFASNET_SOURCE)}
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');use_amp=bool(AMP and device.type=='cuda')
seed_everything(TRAIN_SEED);module=source_module(MINIFASNET_SOURCE)
MINIFASNET_CHECKPOINT=Path(MINIFASNET_CHECKPOINT_PATH)
if not MINIFASNET_CHECKPOINT.is_file():raise FileNotFoundError(MINIFASNET_CHECKPOINT)
initial_model,loading_audit=load_pretrained_canonical(module,MINIFASNET_CHECKPOINT)
resource_hashes['official_checkpoint_sha256']=sha256_file(MINIFASNET_CHECKPOINT)
canonical_state={k:v.detach().cpu().clone() for k,v in initial_model.state_dict().items()}
initial_sha=state_fingerprint(canonical_state)
assert len(canonical_state)>0
strict_json_write(comparison_dir/'resource_fingerprints.json',resource_hashes)
params=sum(p.numel() for p in initial_model.parameters())
if not 300000<=params<=600000:raise RuntimeError('Unexpected MiniFASNetV2 parameter count: '+str(params))
with torch.no_grad():
    initial_model.eval();dummy=initial_model(torch.zeros(2,3,80,80))
    if tuple(dummy.shape)!=(2,3):raise RuntimeError('Model output must be [B,3]')
masks,_=radial_masks(INPUT_SIZE,device);sanity=intervention_sanity(masks,device)
assert selection_by_margins(torch.tensor([[2.,-1.,0.]])).item()==1
assert harmful_margin_loss(torch.tensor([0.]),torch.tensor([1.]))[0].item()==0
assert harmful_margin_loss(torch.tensor([1.]),torch.tensor([0.]))[0].item()>0
probe=new_model(module).to(device).train();before={k:v.clone() for k,v in probe.state_dict().items() if 'running_' in k or 'num_batches_tracked' in k}
select_worst_view(probe,torch.rand(2,3,80,80,device=device),torch.tensor([0,1],device=device),masks,use_amp,2)
assert probe.training and all(torch.equal(v,probe.state_dict()[k]) for k,v in before.items())
del probe,initial_model
strict_json_write(output/'pretrained_loading_audit.json',loading_audit)
strict_json_write(comparison_dir/'initialization_fingerprint.json',{'regime':INIT_NAME,'sha256':initial_sha,
    'params':params,'trainable_params':params,'loading_audit':loading_audit,'sanity':sanity})
print('Frozen resources:',resource_hashes,'init SHA',initial_sha,'parameters',params,'device',device)


In [ ]:
def branch_config(name,csmr_enabled):
    return {'schema':'final_mini_pair_v1','branch':name,'initialization':INIT_NAME,'train_seed':TRAIN_SEED,
        'data_seed':DATA_SEED,'resource_hashes':resource_hashes,'initial_state_sha256':initial_sha,
        'train_manifest':TRAIN_MANIFEST,'val_manifest':VAL_MANIFEST,'test_manifest':'final_test5k',
        'test_role':TEST_ROLE,'checkpoint_selection':'lowest Val ACER; tie highest Val AUC',
        'metric_tie_eps':METRIC_TIE_EPS,
        'hyperparameter_selection':{'source':TUNING_CONFIG_SOURCE,'frozen_hparams_sha256':TUNING_CONFIG_SHA256,
            'selection_stage':'5k_regime_specific_tuning','optimizer_rule':'CelebA Tune-Val ACER then AUC',
            'csmr_rule':'Tune-Val ACER regression guard <= 1.0 pp; LCC AUC then HTER',
            'lcc_role':'external-development only; not untouched final cross-domain test'},
        'input':[3,80,80],'crop_scale':CROP_SCALE,'channels':'BGR','range':'[0,1]',
        'max_epochs':MAX_EPOCHS,'patience':EARLY_STOP_PATIENCE,'earliest_stop':EARLIEST_EARLY_STOP_EPOCH,
        'batch_size':BATCH_SIZE,'amp':use_amp,'optimizer':'AdamW','backbone_lr':BACKBONE_LR,
        'classifier_lr':CLASSIFIER_LR,'weight_decay':WEIGHT_DECAY,'label_smoothing':LABEL_SMOOTHING,
        'grad_clip':GRAD_CLIP,'augmentation':'E1 bbox jitter .2; flip .5; brightness/contrast .3',
        'csmr':{'warmup':CSMR_WARMUP_EPOCHS,'lambda':CSMR_LAMBDA,'centers':CSMR_BAND_CENTERS,
                'sigma':CSMR_BAND_SIGMA,'gain':CSMR_GAIN_RANGE,'preserve_dc':CSMR_PRESERVE_DC}
                if csmr_enabled else None}

def loader_for(frame,train=False,epoch=0):
    dataset=FinalCelebADataset(frame,celeba_root,bbox_cache,train=train)
    gen=torch.Generator().manual_seed(TRAIN_SEED+epoch)
    def worker_init(worker_id):
        value=TRAIN_SEED+epoch*1000+worker_id
        random.seed(value);np.random.seed(value);torch.manual_seed(value)
    return DataLoader(dataset,batch_size=BATCH_SIZE,shuffle=train,drop_last=False,
        num_workers=NUM_WORKERS,pin_memory=device.type=='cuda',generator=gen,worker_init_fn=worker_init)

def run_branch(name,csmr_enabled):
    folder=csmr_dir if csmr_enabled else vanilla_dir
    conf=branch_config(name,csmr_enabled);conf_hash=sha256_text(json.dumps(conf,sort_keys=True))
    config_path=folder/'config.json';last_path=folder/'last_state.pth';best_path=folder/'best.pth';meta_path=folder/'meta.json'
    if config_path.exists():
        if strict_json_load(config_path)!=conf:raise RuntimeError(f'{name} config or checkpoint policy differs; restart paired run in a clean output directory')
    else:strict_json_write(config_path,conf)
    seed_everything(TRAIN_SEED)
    model=new_model(module);model.load_state_dict(canonical_state,strict=True)
    if state_fingerprint(model.state_dict())!=initial_sha:raise RuntimeError('Paired initial state drift')
    model=model.to(device)
    if INIT_NAME=='pretrained':
        backbone=[p for n,p in model.named_parameters() if not n.startswith('prob.')]
        groups=[{'params':backbone,'lr':BACKBONE_LR},{'params':list(model.prob.parameters()),'lr':CLASSIFIER_LR}]
    else:groups=[{'params':model.parameters(),'lr':BACKBONE_LR}]
    optimizer=torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)
    scheduler=None
    scaler=torch.cuda.amp.GradScaler(enabled=use_amp)
    criterion=nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    history=[];diagnostics=[];class_rows=[];best_epoch=0;best_acer=float('inf');best_auc=-float('inf');start_epoch=1;resume_stopped=False
    if meta_path.exists():
        meta=strict_json_load(meta_path)
        if meta['config_hash']!=conf_hash or not best_path.is_file():raise RuntimeError('Completed branch provenance mismatch')
        checkpoint=torch.load(best_path,map_location='cpu',weights_only=True)
        if sha256_file(best_path)!=meta['checkpoint_sha256']:raise RuntimeError('Completed best SHA mismatch')
        if 'best_val_acer' not in meta or 'best_val_auc' not in meta:
            raise RuntimeError('Completed metadata predates Val AUC tie-break; restart this branch')
        if (checkpoint.get('epoch')!=meta['selected_epoch'] or
            abs(checkpoint.get('best_val_acer',float('inf'))-meta['best_val_acer'])>METRIC_VERIFY_EPS or
            abs(checkpoint.get('best_val_auc',-float('inf'))-meta['best_val_auc'])>METRIC_VERIFY_EPS):
            raise RuntimeError('Completed best checkpoint/metadata selection mismatch')
        model.load_state_dict(checkpoint['state_dict'])
        logits,y,keys,_=eval_spatial(model.to(device),loader_for(val_frame),device,use_amp)
        assert_ordered_keys(keys,val_frame.sample_key.astype(str).tolist(),'Completed frozen Val')
        threshold=threshold_from_val(y,binary_score(torch.from_numpy(logits)).numpy())
        verified,_=metrics_celeba(y,logits,threshold)
        if (abs(verified['acer']-meta['best_val_acer'])>METRIC_VERIFY_EPS or
            abs(verified['auc']-meta['best_val_auc'])>METRIC_VERIFY_EPS):
            raise RuntimeError('Completed best checkpoint fails frozen Val metric verification')
        model.cpu()
        return {'name':name,'model':model,'folder':folder,'val':verified,
                'locked':meta['locked_val_threshold'],'epoch':meta['selected_epoch'],
                'best_acer':meta['best_val_acer'],'best_auc':meta['best_val_auc'],
                'sha':meta['checkpoint_sha256'],'initial_sha':initial_sha}
    if last_path.exists():
        if not RESUME:raise RuntimeError(f'{name} incomplete state exists; set RESUME=True or use clean output dir')
        state=torch.load(last_path,map_location='cpu',weights_only=False)
        if state['config_hash']!=conf_hash or state['branch']!=name or state['initial_state_sha256']!=initial_sha:
            raise RuntimeError('Resume provenance mismatch')
        model.load_state_dict(state['model_state']);optimizer.load_state_dict(state['optimizer_state'])
        scaler.load_state_dict(state['scaler_state'])
        history=state['history'];diagnostics=state['diagnostics'];class_rows=state['class_diagnostics']
        if 'best_auc' not in state:raise RuntimeError('Old resume state lacks best_auc; restart paired run')
        best_epoch=state['best_epoch'];best_acer=state['best_acer'];best_auc=state['best_auc'];start_epoch=state['epoch']+1
        resume_stopped=state.get('stop_reached',False)
        random.setstate(state['rng_python']);np.random.set_state(state['rng_numpy']);torch.set_rng_state(state['rng_torch'])
        if device.type=='cuda' and state['rng_cuda'] is not None:torch.cuda.set_rng_state_all(state['rng_cuda'])
        if not best_path.is_file():raise RuntimeError('Resume best checkpoint missing')
    val_loader=loader_for(val_frame)
    try:
        for epoch in ([] if resume_stopped else range(start_epoch,MAX_EPOCHS+1)):
            started=time.monotonic();model.train();train_loader=loader_for(train_frame,train=True,epoch=epoch)
            losses=[];ce_clean=[];ce_worst=[];cls_losses=[];margin_losses=[]
            band_count=torch.zeros(3,dtype=torch.int64);class_band=torch.zeros((3,3),dtype=torch.int64)
            class_n=torch.zeros(3,dtype=torch.int64);class_harm=torch.zeros(3,dtype=torch.int64)
            class_drop=torch.zeros(3,dtype=torch.float64)
            n_csmr=0;harm_n=0;harm_drop=clean_sum=worst_sum=gain_sum=clip_sum=0.
            print(f'{name} epoch {epoch}/{MAX_EPOCHS}',flush=True)
            for bi,(x,y,_,_) in enumerate(train_loader,1):
                x=x.to(device,non_blocking=True);y=y.to(device,non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                if csmr_enabled and epoch>CSMR_WARMUP_EPOCHS:
                    selected,band_idx,gains,clips=select_worst_view(model,x,y,masks,use_amp,SELECTION_CHUNK_SIZE)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(torch.cat([x,selected],dim=0));clean_logits,worst_logits=logits.chunk(2)
                        clean_ce=criterion(clean_logits,y);worst_ce=criterion(worst_logits,y)
                        cls_loss=.5*clean_ce+.5*worst_ce
                        clean_margin=aligned_margin(clean_logits,y);worst_margin=aligned_margin(worst_logits,y)
                        margin_loss,harmful,drop=harmful_margin_loss(clean_margin,worst_margin)
                        loss=cls_loss+CSMR_LAMBDA*margin_loss
                    n=len(y);n_csmr+=n;harm_n+=int(harmful.sum().cpu())
                    harm_drop+=float(drop.clamp_min(0).sum().detach().cpu());clean_sum+=float(clean_margin.sum().detach().cpu())
                    worst_sum+=float(worst_margin.sum().detach().cpu());gain_sum+=float(gains.sum().cpu());clip_sum+=float(clips.sum().cpu())
                    ce_clean.append(float(clean_ce.detach().cpu()));ce_worst.append(float(worst_ce.detach().cpu()))
                    for b in range(3):band_count[b]+=int((band_idx==b).sum().cpu())
                    for cls in range(3):
                        mask=y==cls;class_n[cls]+=int(mask.sum().cpu());class_harm[cls]+=int((harmful&mask).sum().cpu())
                        class_drop[cls]+=float(drop.clamp_min(0)[mask].sum().detach().cpu())
                        for b in range(3):class_band[cls,b]+=int((mask&(band_idx==b)).sum().cpu())
                else:
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits=model(x);cls_loss=criterion(logits,y);margin_loss=cls_loss.detach()*0;loss=cls_loss
                    ce_clean.append(float(cls_loss.detach().cpu()))
                if not torch.isfinite(loss):raise RuntimeError('Nonfinite train loss')
                scaler.scale(loss).backward();scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
                scaler.step(optimizer);scaler.update()
                losses.append(float(loss.detach().cpu()));cls_losses.append(float(cls_loss.detach().cpu()))
                margin_losses.append(float(margin_loss.detach().cpu()))
                if bi==1 or bi%max(1,len(train_loader)//10)==0:print(f'  batch {bi}/{len(train_loader)} loss {np.mean(losses):.4f}',flush=True)
            val_logits,val_y,val_keys,_=eval_spatial(model,val_loader,device,use_amp)
            assert_ordered_keys(val_keys,val_frame.sample_key.astype(str).tolist(),'Val')
            val_score=binary_score(torch.from_numpy(val_logits)).numpy();threshold=threshold_from_val(val_y,val_score)
            val_metrics,_=metrics_celeba(val_y,val_logits,threshold)
            history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),'val_acer':val_metrics['acer'],
                'val_auc':val_metrics['auc'],'val_threshold':threshold,'seconds':time.monotonic()-started})
            if csmr_enabled:
                d={'epoch':epoch,'ce_clean':float(np.mean(ce_clean)),'ce_worst':float(np.mean(ce_worst)) if ce_worst else None,
                   'classification_loss':float(np.mean(cls_losses)),'margin_loss':float(np.mean(margin_losses)),
                   'total_loss':float(np.mean(losses)),'harmful_fraction':harm_n/n_csmr if n_csmr else None,
                   'mean_harmful_margin_drop':harm_drop/harm_n if harm_n else None,
                   'mean_clean_aligned_margin':clean_sum/n_csmr if n_csmr else None,
                   'mean_worst_aligned_margin':worst_sum/n_csmr if n_csmr else None,
                   'mean_gain':gain_sum/n_csmr if n_csmr else None,'clipping_fraction':clip_sum/n_csmr if n_csmr else None}
                for b in range(3):d['worst_'+BANDS[b].lower()+'_fraction']=int(band_count[b])/n_csmr if n_csmr else None
                diagnostics.append(d)
                for cls,label in enumerate(('Real','Physical','Digital')):
                    n=int(class_n[cls]);cr={'epoch':epoch,'class':label,'n':n,
                        'harmful_fraction':int(class_harm[cls])/n if n else None,
                        'mean_harmful_margin_drop':float(class_drop[cls])/int(class_harm[cls]) if class_harm[cls] else None}
                    for b in range(3):cr['worst_'+BANDS[b].lower()+'_fraction']=int(class_band[cls,b])/n if n else None
                    class_rows.append(cr)
            val_acer=float(val_metrics['acer']);val_auc=float(val_metrics['auc'])
            better=(val_acer<best_acer-METRIC_TIE_EPS or
                (abs(val_acer-best_acer)<=METRIC_TIE_EPS and val_auc>best_auc+METRIC_TIE_EPS))
            if better:
                best_acer=val_acer;best_auc=val_auc;best_epoch=epoch
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                torch.save({'state_dict':best_state,'epoch':epoch,
                    'best_val_acer':best_acer,'best_val_auc':best_auc},best_path.with_suffix('.pth.tmp'))
                os.replace(best_path.with_suffix('.pth.tmp'),best_path)
            state={'branch':name,'config_hash':conf_hash,'initial_state_sha256':initial_sha,'epoch':epoch,
                'model_state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'optimizer_state':optimizer.state_dict(),'scheduler_state':scheduler.state_dict() if scheduler else None,
                'scaler_state':scaler.state_dict(),'best_epoch':best_epoch,'best_acer':best_acer,'best_auc':best_auc,
                'history':history,'diagnostics':diagnostics,'class_diagnostics':class_rows,
                'rng_python':random.getstate(),'rng_numpy':np.random.get_state(),'rng_torch':torch.get_rng_state(),
                'rng_cuda':torch.cuda.get_rng_state_all() if device.type=='cuda' else None,
                'stop_reached':bool(epoch==MAX_EPOCHS or (epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE))}
            torch.save(state,last_path.with_suffix('.pth.tmp'))
            os.replace(last_path.with_suffix('.pth.tmp'),last_path)
            print(f'  Val ACER {val_metrics["acer"]:.6f}, AUC {val_metrics["auc"]:.6f}; best epoch {best_epoch}, best ACER {best_acer:.6f}, best AUC {best_auc:.6f}',flush=True)
            if epoch>=EARLIEST_EARLY_STOP_EPOCH and epoch-best_epoch>=EARLY_STOP_PATIENCE:break
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError('OOM: restart both branches from clean output with BATCH_SIZE=64') from exc
    if not best_path.is_file():raise RuntimeError('No Val-selected best checkpoint')
    checkpoint=torch.load(best_path,map_location='cpu',weights_only=True)
    if (checkpoint.get('epoch')!=best_epoch or
        abs(checkpoint.get('best_val_acer',float('inf'))-best_acer)>METRIC_VERIFY_EPS or
        abs(checkpoint.get('best_val_auc',-float('inf'))-best_auc)>METRIC_VERIFY_EPS):
        raise RuntimeError('Saved best checkpoint selection metadata mismatch')
    model.load_state_dict(checkpoint['state_dict'])
    logits,y,keys,codes=eval_spatial(model,val_loader,device,use_amp)
    assert_ordered_keys(keys,val_frame.sample_key.astype(str).tolist(),'Frozen Val')
    locked=threshold_from_val(y,binary_score(torch.from_numpy(logits)).numpy())
    metrics,_=metrics_celeba(y,logits,locked)
    if (abs(metrics['acer']-best_acer)>METRIC_VERIFY_EPS or
        abs(metrics['auc']-best_auc)>METRIC_VERIFY_EPS):
        raise RuntimeError('Reloaded best checkpoint fails Val ACER/AUC verification')
    print(f'FROZEN {name}: best_epoch={best_epoch}, best_val_acer={best_acer:.8f}, best_val_auc={best_auc:.8f}')
    make_prediction_frame(keys,y,codes,logits,locked).to_csv(folder/'val_predictions.csv',index=False)
    pd.DataFrame(history).to_csv(folder/'training_history.csv',index=False)
    if csmr_enabled:
        pd.DataFrame(diagnostics).to_csv(folder/'csmr_diagnostics.csv',index=False)
        pd.DataFrame(class_rows).to_csv(folder/'class_diagnostics.csv',index=False)
    meta={'branch':name,'config_hash':conf_hash,'selected_epoch':best_epoch,'checkpoint_sha256':sha256_file(best_path),
        'initial_state_sha256':initial_sha,'locked_val_threshold':locked,'threshold_source':'CelebA Val only',
        'best_val_acer':best_acer,'best_val_auc':best_auc,'test_role':TEST_ROLE,'val':metrics}
    strict_json_write(meta_path,meta)
    model.cpu();
    if device.type=='cuda':torch.cuda.empty_cache()
    return {'name':name,'model':model,'folder':folder,'val':metrics,'locked':locked,'epoch':best_epoch,
        'sha':meta['checkpoint_sha256'],'initial_sha':initial_sha,
        'best_acer':best_acer,'best_auc':best_auc}

vanilla=run_branch('vanilla',False)
csmr=run_branch('csmr',True)
assert vanilla['initial_sha']==csmr['initial_sha']==initial_sha
strict_json_write(comparison_dir/'frozen_development.json',{
    'vanilla':{k:vanilla[k] for k in ('epoch','sha','locked','val','best_acer','best_auc')},
    'csmr':{k:csmr[k] for k in ('epoch','sha','locked','val','best_acer','best_auc')},
    'initial_sha256':initial_sha,'test_role':TEST_ROLE,
    'policy':'Both best checkpoints and thresholds locked from Val before Test or LCC access'})
print('FROZEN development:',initial_sha,vanilla['sha'],csmr['sha'])


In [ ]:
if not (comparison_dir/'frozen_development.json').is_file():raise RuntimeError('Freeze decision not saved')

def evaluate_pair_split(split,frame):
    loader=loader_for(frame)
    out=[]
    for result in (vanilla,csmr):
        model=result['model'].to(device)
        logits,y,keys,codes=eval_spatial(model,loader,device,use_amp)
        assert_ordered_keys(keys,frame.sample_key.astype(str).tolist(),split)
        metrics,_=metrics_celeba(y,logits,result['locked'])
        pred=make_prediction_frame(keys,y,codes,logits,result['locked'])
        pred.to_csv(result['folder']/(split+'_predictions.csv'),index=False)
        cm=confusion_matrix(y,np.argmax(logits,axis=1),labels=[0,1,2])
        strict_json_write(result['folder']/(split+'_confusion.json'),{'labels':['Real','Physical','Digital'],'matrix':cm.tolist()})
        for cls,label in enumerate(('Real','Physical','Digital')):
            recall=float(cm[cls,cls]/cm[cls].sum())
            class_rows.append({'split':split,'branch':result['name'],'class':label,'n':int(cm[cls].sum()),
                'recall':recall,'error':1-recall})
        result[split]=metrics;out.append({'branch':result['name'],**metrics})
        model.cpu()
        if device.type=='cuda':torch.cuda.empty_cache()
    pd.DataFrame(out).to_csv(comparison_dir/(split+'_metrics.csv'),index=False)
    return out
class_rows=[]
val_rows=evaluate_pair_split('val',val_frame)
test_rows=evaluate_pair_split('test',test)
lcc_dataset=FinalLCCDataset(lcc_frame,lcc_root,lcc_cache)
lcc_loader=DataLoader(lcc_dataset,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=device.type=='cuda')
lcc_rows=[]
for result in (vanilla,csmr):
    model=result['model'].to(device)
    logits,y,keys,frame=eval_lcc_spatial(model,lcc_loader,device,use_amp,lcc_frame,result['locked'])
    score=binary_score(torch.from_numpy(logits)).numpy()
    metrics=metrics_binary(y,score,result['locked'],coverage=len(y)/len(lcc_frame),candidate_n=len(lcc_frame))
    frame.to_csv(result['folder']/'lcc_predictions.csv',index=False)
    result['lcc']=metrics;lcc_rows.append({'branch':result['name'],**metrics})
    model.cpu()
    if device.type=='cuda':torch.cuda.empty_cache()
pd.DataFrame(lcc_rows).to_csv(comparison_dir/'lcc_metrics.csv',index=False)
pd.DataFrame(class_rows).to_csv(comparison_dir/'class_metrics.csv',index=False)
rows=[]
for split in ('val','test','lcc'):
    for metric in ('apcer','bpcer','acer','auc')+ (('hter',) if split=='lcc' else ()):
        a=vanilla[split][metric];b=csmr[split][metric]
        rows.append({'split':split,'metric':metric,'vanilla':a,'csmr':b,'delta_csmr_minus_vanilla':b-a})
pd.DataFrame(rows).to_csv(comparison_dir/'paired_metrics.csv',index=False)
for split in ('val','test','lcc'):
    fig,ax=plt.subplots(figsize=(5,5))
    for result in (vanilla,csmr):
        pred=pd.read_csv(result['folder']/(split+'_predictions.csv' if split!='lcc' else 'lcc_predictions.csv'))
        if split=='lcc':pred=pred[pred.status=='VALID'];score=pred.pad_logit_score.to_numpy();real=(pred.label.to_numpy()==0)
        else:score=pred.pad_logit_score.to_numpy();real=(pred.true_class.to_numpy()==0)
        fpr,tpr,_=roc_curve(real.astype(int),score);ax.plot(fpr,tpr,label=result['name'])
    ax.plot([0,1],[0,1],'--',color='gray');ax.set(xlabel='FPR',ylabel='TPR',title=split+' ROC');ax.legend()
    fig.tight_layout();fig.savefig(comparison_dir/('roc_'+split+'.png'),dpi=140);plt.close(fig)
print(pd.DataFrame(rows).to_string(index=False))

# Compact primary table for the 20K confirmation.
primary=[row for row in rows if (row['split'],row['metric']) in
    {('val','acer'),('val','auc'),('test','acer'),('test','auc'),('lcc','hter'),('lcc','auc')}]
pd.DataFrame(primary,columns=['split','metric','vanilla','csmr','delta_csmr_minus_vanilla']).to_csv(
    comparison_dir/'metrics_comparison.csv',index=False)
for branch in (vanilla,csmr):
    strict_json_write(branch['folder']/'metrics.json',{
        'schema':'minifasnetv2_scale20k_pretrained_branch_v1',
        'checkpoint_sha256':branch['sha'],'initial_state_sha256':initial_sha,
        'selected_epoch':branch['epoch'],'locked_val_threshold':branch['locked'],
        'val':branch['val'],'test':branch['test'],'lcc':branch['lcc'],
        'test_role':TEST_ROLE,'lcc_role':'external-development; used for CSMR selection in 00b-v2'})


In [ ]:
lines=['# MiniFASNetV2 pretrained 20K scale confirmation','',
    '## Objective','Compare pretrained Vanilla with the pretrained-specific CSMR recipe frozen by Notebook 00b-v2. No learning rate, lambda, gain, threshold, or checkpoint is selected from Test-5K or LCC here.','',
    '## Initialization',f'Official PAD-pretrained feature weights plus deterministic fresh 3-class classifier. Loading audit: `pretrained_loading_audit.json`. Both branches start from canonical state SHA256 `{initial_sha}`.','',
    '## Frozen resources',f'Train {len(train_frame)}, Val {len(val_frame)}, Test {len(test)}, LCC candidates {len(lcc_frame)} and valid {len(lcc_dataset)}. Resource hashes: `{resource_hashes}`.','',
    '## Model and preprocessing',f'MiniFASNetV2, conv6 5x5, 3 logits, {params} trainable parameters; 2.7x face crop, 80x80 BGR float [0,1].','',
    '## Frozen v2 recipe',f'Selection SHA256 `{TUNING_CONFIG_SHA256}`. AdamW backbone/head LR {BACKBONE_LR}/{CLASSIFIER_LR}; CSMR lambda {CSMR_LAMBDA}, gains {CSMR_GAIN_RANGE}, centers {CSMR_BAND_CENTERS}, sigma {CSMR_BAND_SIGMA}, warmup {CSMR_WARMUP_EPOCHS}, preserve DC {CSMR_PRESERVE_DC}.','',
    '## Training budget',f'Seed {TRAIN_SEED}; batch {BATCH_SIZE}; AMP {use_amp}; maximum {MAX_EPOCHS} epochs; patience {EARLY_STOP_PATIENCE}; earliest stop epoch {EARLIEST_EARLY_STOP_EPOCH}. Both branches use the same optimizer recipe and canonical initial state.','',
    '## Frozen Val decisions',f'Vanilla epoch {vanilla["epoch"]}, Val ACER {vanilla["best_acer"]:.6f}, Val AUC {vanilla["best_auc"]:.6f}, threshold {vanilla["locked"]:.6f}, checkpoint `{vanilla["sha"]}`.',
    f'CSMR epoch {csmr["epoch"]}, Val ACER {csmr["best_acer"]:.6f}, Val AUC {csmr["best_auc"]:.6f}, threshold {csmr["locked"]:.6f}, checkpoint `{csmr["sha"]}`.','',
    '## Primary paired metrics','| Split | Metric | Vanilla | CSMR | Delta CSMR minus Vanilla |',
    '|---|---|---:|---:|---:|']
for row in primary:
    lines.append(f'| {row["split"]} | {row["metric"]} | {row["vanilla"]:.6f} | {row["csmr"]:.6f} | {row["delta_csmr_minus_vanilla"]:+.6f} |')
lines+=['','## Diagnostics','Class metrics and confusion matrices are saved under comparison/ and each branch. CSMR training diagnostics are saved under csmr/. Inference uses the ordinary MiniFASNetV2 graph.','',
    '## Evaluation roles','Test-5K is fixed in-domain descriptive evaluation. It did not influence hyperparameters, checkpoint selection, or threshold calibration. LCC-FASD is external-development stress and was used in 00b-v2 for CSMR selection, so it is not an untouched final external test. CASIA-FASD remains untouched.','',
    '## Limitations','This single-seed 20K comparison confirms scale behavior; it does not estimate multi-seed uncertainty.']
(comparison_dir/'scale20k_pretrained_report.md').write_text('\n'.join(lines)+'\n')


In [ ]:
ZIP_PATH=Path('/kaggle/working/minifasnet_scale20k_pretrained_seed100.zip')
write_zip(output,ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH) as archive:
    names=set(archive.namelist())
    for relative in ('pretrained_loading_audit.json','vanilla/best.pth','csmr/best.pth','comparison/metrics_comparison.csv','comparison/class_metrics.csv','comparison/scale20k_pretrained_report.md'):
        if output.name+'/'+relative not in names:raise RuntimeError('Missing from ZIP: '+relative)
print('ZIP:',ZIP_PATH,'size MB:',ZIP_PATH.stat().st_size/1e6)
display(FileLink(str(ZIP_PATH)))
